In [1]:
import pandas as pd
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()

DB_USER = os.getenv("DB_USER")
DB_PASS = os.getenv("DB_PASS")
DB_HOST = os.getenv("DB_HOST")
DB_NAME = os.getenv("DB_NAME")

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:5432/{DB_NAME}"
)

In [2]:
df = pd.read_sql("SELECT * FROM customer_behavior_raw", engine)
df.head()

,user_id,age,gender,country,urban_rural,income_level,employment_status,education_level,relationship_status,has_children,...,cart_items_average,checkout_abandonments_per_month,purchase_conversion_rate,app_usage_frequency,notification_response_rate,account_age_months,last_purchase_date,social_sharing_frequency,premium_subscription,return_rate
0,1,56,Female,Germany,Suburban,90860,Self-employed,Associate Degree,Single,0,...,10,2,62,7,74,19,2025-06-22,6,1,50
1,2,69,Male,Japan,Suburban,35423,Unemployed,Bachelor,Single,1,...,5,7,54,5,23,8,2026-07-25,3,0,37
2,3,46,Female,India,Urban,21467,Self-employed,Associate Degree,Married,1,...,3,3,33,7,12,13,2026-02-26,6,0,53
3,4,32,Male,Canada,Urban,41770,Self-employed,Bachelor,Widowed,0,...,5,9,26,4,19,9,2026-10-27,7,0,98
4,5,60,Female,Japan,Urban,183882,Employed,Associate Degree,Widowed,1,...,8,0,18,7,30,3,2026-06-23,3,0,86


In [3]:
print("Shape:", df.shape)
df.info()

Shape: (1000000, 60)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 60 columns):
 #   Column                           Non-Null Count    Dtype 
---  ------                           --------------    ----- 
 0   user_id                          1000000 non-null  int64 
 1   age                              1000000 non-null  int64 
 2   gender                           1000000 non-null  object
 3   country                          1000000 non-null  object
 4   urban_rural                      1000000 non-null  object
 5   income_level                     1000000 non-null  int64 
 6   employment_status                1000000 non-null  object
 7   education_level                  1000000 non-null  object
 8   relationship_status              1000000 non-null  object
 9   has_children                     1000000 non-null  int64 
 10  household_size                   1000000 non-null  int64 
 11  occupation                       1000000 no

In [4]:
df.isnull().sum().sort_values(ascending=False).head(10)

user_id                         0
age                             0
impulse_buying_score            0
environmental_consciousness     0
health_conscious_shopping       0
travel_frequency                0
hobby_count                     0
social_media_influence_score    0
reading_habits                  0
exercise_frequency              0
dtype: int64

In [5]:
# Clean column names
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

df.columns


Index(['user_id', 'age', 'gender', 'country', 'urban_rural', 'income_level',
       'employment_status', 'education_level', 'relationship_status',
       'has_children', 'household_size', 'occupation', 'ethnicity',
       'language_preference', 'device_type', 'weekly_purchases',
       'monthly_spend', 'cart_abandonment_rate', 'review_writing_frequency',
       'average_order_value', 'preferred_payment_method',
       'coupon_usage_frequency', 'loyalty_program_member', 'referral_count',
       'product_category_preference', 'shopping_time_of_day',
       'weekend_shopper', 'impulse_purchases_per_month', 'browse_to_buy_ratio',
       'return_frequency', 'budgeting_style', 'brand_loyalty_score',
       'impulse_buying_score', 'environmental_consciousness',
       'health_conscious_shopping', 'travel_frequency', 'hobby_count',
       'social_media_influence_score', 'reading_habits', 'exercise_frequency',
       'stress_from_financial_decisions', 'overall_stress_level',
       'sleep_quali

In [6]:

num_cols = [
    'age', 'monthly_spend', 'average_order_value',
    'cart_abandonment_rate', 'browse_to_buy_ratio'
]

df[num_cols] = df[num_cols].apply(pd.to_numeric, errors='coerce')


In [7]:
cat_cols = df.select_dtypes(include='object').columns

df[cat_cols] = df[cat_cols].apply(lambda x: x.str.strip().str.lower())

In [8]:
df.fillna(0, inplace=True)

In [9]:

df= df.drop_duplicates()

In [10]:
df["age_group"] = pd.cut(
    df["age"],
    bins=[18, 25, 35, 50, 70],
    labels=["18-25", "26-35", "36-50", "51+"]
)


In [11]:
df["income_band"] = pd.qcut(
    df["income_level"],
    q=3,
    labels=["low", "medium", "high"]
)


In [12]:
df['spend_level'] = pd.cut(
    df['monthly_spend'],
    bins=[0, 2000, 5000, 10000],
    labels=['Low', 'Medium', 'High']
)


In [ ]:
df.to_sql(
    "customer_behavior_cleaned",
    engine,
    if_exists="replace",
    index=False
)

print("customer_behavior_cleaned table created ✅")
